# Chess Engine Project - Analysis File
In this file I conduct various types of analyses. The outcomes are used in the presentation.

In [1]:
# import project files
import games
import players
import utils
import players_history

# import other libraries
import chess
import time

### 1. SimpleEngine tournament
In the next cell I let the three heuristics in the SimpleEngine class play against each other in a tournament.

In [6]:
player_1 = players.SimpleEngine('random')
player_2 = players.SimpleEngine('attacking')
player_3 = players.SimpleEngine('limiting')

player_list = [player_1, player_2, player_3]

tournament_results = games.tournament(player_list, num_games=100)

Match: randomEngine vs randomEngine: [7, 82, 11, 0]
Match: randomEngine vs attackingEngine: [0, 79, 21, 0]
Match: randomEngine vs limitingEngine: [1, 51, 48, 0]
Match: attackingEngine vs randomEngine: [30, 69, 1, 0]
Match: attackingEngine vs attackingEngine: [7, 89, 4, 0]
Match: attackingEngine vs limitingEngine: [6, 81, 13, 0]
Match: limitingEngine vs randomEngine: [43, 57, 0, 0]
Match: limitingEngine vs attackingEngine: [9, 85, 6, 0]
Match: limitingEngine vs limitingEngine: [4, 84, 12, 0]
white_results {'randomEngine': [8, 212, 80, 0], 'attackingEngine': [43, 239, 18, 0], 'limitingEngine': [56, 226, 18, 0]}
black_results {'randomEngine': [80, 208, 12, 0], 'attackingEngine': [16, 253, 31, 0], 'limitingEngine': [11, 216, 73, 0]}
total_results {'randomEngine': [20, 420, 160, 0], 'attackingEngine': [74, 492, 34, 0], 'limitingEngine': [129, 442, 29, 0]}


### 2. Evaluation function - Speed test
In the next cells I test the speed of the evaluation function with and without the use of numba (where applicable).

In [4]:
n = 100_000
print(f'Speed tests for {n} calls:')

board = chess.Board()


# speed tests without Numba

# test speed of total evaluation function
t0 = time.perf_counter()
for i in range(n):
    evaluation = utils.evaluate_board_nonumba(board)
t1 = time.perf_counter()
print('evaluate_board_nonumba(board):',t1-t0)

# test speed of first part of total evaluation function
t0 = time.perf_counter()
for i in range(n):
    evaluation = utils.convert_to_numeric(board)
t1 = time.perf_counter()
print('convert_to_numeric(board):',t1-t0)

# test speed of second part of total evaluation function
numeric_board = utils.convert_to_numeric(board)
t0 = time.perf_counter()
for i in range(n):
    evaluation = utils.compute_evaluation_nonumba(numeric_board)
t1 = time.perf_counter()
print('compute_evaluation_nonumba(numeric_board):',t1-t0)


# speed tests with Numba:
evaluation = utils.evaluate_board(board) # call once so that Numba is initiated

# test speed of total evaluation function
t0 = time.perf_counter()
for i in range(n):
    evaluation = utils.evaluate_board(board)
t1 = time.perf_counter()
print('evaluate_board(board):',t1-t0)

# Numba is not appropriate for first part of total evaluation function

# test speed of second part of total evaluation function
numeric_board = utils.convert_to_numeric(board)
t0 = time.perf_counter()
for i in range(n):
    evaluation = utils.compute_evaluation(numeric_board)
t1 = time.perf_counter()
print('compute_evaluation(numericboard):',t1-t0)

Speed tests for 100000 calls:
evaluate_board_nonumba(board): 7.744935599999991
convert_to_numeric(board): 4.162994600000005
compute_evaluation_nonumba(numeric_board): 2.6910859999999985
evaluate_board(board): 4.468905300000003
compute_evaluation(numericboard): 0.28065789999999424


### 3. Pruning and move-ordering analysis
In the next cells I analyze the reduction of the necessary number evaluations for alpha-beta pruning with different mthods of move-ordering.

In [2]:
def compare_versions(depth, board):
#     player1 = players_history.MinimaxEngine(depth=depth)
    player2 = players_history.NegamaxEngineV1(depth=depth)
#     player3 = players_history.NegamaxEngineV2(depth=depth)
#     player4 = players_history.NegamaxEngineV3(depth=depth)
#     player5 = players_history.NegamaxEngineV4(depth=depth)
    player6 = players_history.NegamaxEngineV5(depth=depth)
    player7 = players_history.NegamaxEngineV6(depth=depth)
    player8 = players_history.NegamaxEngineV7(depth=depth)
    player_list = [player2, player6, player7, player8]
    
    for player in player_list:
        t0 = time.perf_counter()
        move = player.make_move(board)
        t1 = time.perf_counter()
        print(f'name:',str(player),'move:',move,'time:',t1-t0,'num_evals:', player.num_evals)

In [4]:
# exemplary position
board=chess.Board('r3k2r/pbp1bp2/2nqpp2/1p1p3p/P3P1QP/1PNP1N2/2P1BPP1/R3K2R b KQkq - 0 11')

print('\nDepth=1')
compare_versions(depth=1, board=board)

print('\nDepth=2')
compare_versions(depth=2, board=board)

print('\nDepth=3')
compare_versions(depth=3, board=board)

print('\nDepth=4')
compare_versions(depth=4, board=board)


Depth=1
name: NegamaxEngineV1(depth=1,no_pruning) move: h5g4 time: 0.007327700000018922 num_evals: 39
name: NegamaxEngineV5(depth=1,ab_pruning,no_ordering) move: h5g4 time: 0.004914700000000494 num_evals: 39
name: NegamaxEngineV6(depth=1,ab_pruning,random_ordering) move: h5g4 time: 0.005063299999989113 num_evals: 39
name: NegamaxEngineV7(depth=1,ab_pruning,systematic_ordering) move: h5g4 time: 0.004692900000009104 num_evals: 39

Depth=2
name: NegamaxEngineV1(depth=2,no_pruning) move: h5g4 time: 0.17647420000000125 num_evals: 1689
name: NegamaxEngineV5(depth=2,ab_pruning,no_ordering) move: h5g4 time: 0.0347116000000085 num_evals: 243
name: NegamaxEngineV6(depth=2,ab_pruning,random_ordering) move: h5g4 time: 0.03241030000000933 num_evals: 166
name: NegamaxEngineV7(depth=2,ab_pruning,systematic_ordering) move: h5g4 time: 0.02963339999999448 num_evals: 117

Depth=3
name: NegamaxEngineV1(depth=3,no_pruning) move: h5g4 time: 6.996699300000017 num_evals: 63455
name: NegamaxEngineV5(depth=3,a